In [ ]:
import torch.nn as nn
import torch
import numpy
from matplotlib import pyplot as plt
from datasets import load_dataset
from huggingface_hub import login
from torchvision import transforms
from torch.utils.data import DataLoader, IterableDataset
from tqdm import tqdm
from google.colab import drive

login()
drive.mount('/content/drive')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
batch_size = 64

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class StreamingImageNet(IterableDataset):
    def __init__(self, hf_dataset, transform):
        self.ds = hf_dataset
        self.transform = transform
    def __iter__(self):
        for s in self.ds:
            yield self.transform(s["image"].convert("RGB")), s["label"]

DS = load_dataset("imagenet-1k", split="train", streaming=True).shuffle(seed=42, buffer_size=1000)
dataset = StreamingImageNet(DS, transform)
loader = DataLoader(dataset, batch_size=batch_size, num_workers=4, pin_memory=True)

In [ ]:
initial = nn.Sequential(
    nn.Conv2d(in_channels=3, out_channels=96, stride=2, kernel_size=(7,7), padding=3),
    nn.BatchNorm2d(num_features=96),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=(3,3), stride=2, padding=1)
)
k = 48

In [ ]:
class DenseBlock(nn.Module):
    def __init__(self, k, layers, in_layer):
        super().__init__()
        self.layers = nn.ModuleList([])
        for i in range(layers):
            self.layers.append(nn.Sequential(
                nn.BatchNorm2d(num_features=in_layer + i*k),
                nn.ReLU(),
                nn.Conv2d(in_layer + i*k, 4*k, kernel_size=(1,1)),
                nn.BatchNorm2d(4*k),
                nn.ReLU(),
                nn.Conv2d(4*k, k, kernel_size=(3,3), padding=1)
            ))
    def forward(self, x: torch.Tensor):
        for layer in self.layers:
            out = layer(x)
            x = torch.concat([x, out], dim=1)
        return x

In [ ]:
class Transition(nn.Module):
    def __init__(self, i):
        super().__init__()
        self.l1 = nn.Conv2d(i, i//2, kernel_size=(1,1))
        self.l2 = nn.AvgPool2d(stride=2, kernel_size=(2,2))
    def forward(self, x):
        return self.l2(self.l1(x))

In [ ]:
c0 = 96
c1 = c0 + 6*k
c2 = c1//2 + 12*k
c3 = c2//2 + 36*k
c4 = c3//2 + 24*k

DenseNet161 = nn.Sequential(
    initial,
    DenseBlock(k, 6,  c0),
    Transition(c1),
    DenseBlock(k, 12, c1//2),
    Transition(c2),
    DenseBlock(k, 36, c2//2),
    Transition(c3),
    DenseBlock(k, 24, c3//2),
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Linear(c4, 1000)
).to(device)

In [ ]:
CHECKPOINT_PATH = "/content/drive/MyDrive/densenet161_checkpoint.pt"

epochs = 3
optimizer = torch.optim.Adam(DenseNet161.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()
start_epoch = 0

try:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    DenseNet161.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"]
    print(f"Resumed from epoch {start_epoch}")
except FileNotFoundError:
    print("No checkpoint found, starting fresh")

In [ ]:
IMAGENET_SIZE = 1281167
batches_per_epoch = IMAGENET_SIZE // batch_size

for epoch in range(start_epoch, epochs):
    for batch_idx, (imgs, lbls) in enumerate(tqdm(loader, total=batches_per_epoch, desc=f"Epoch {epoch+1}/{epochs}")):
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        out = DenseNet161(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()

        if batch_idx % 500 == 0 and batch_idx > 0:
            torch.save({
                "epoch": epoch,
                "batch_idx": batch_idx,
                "model_state_dict": DenseNet161.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "loss": loss.item(),
            }, CHECKPOINT_PATH)
            tqdm.write(f"Epoch {epoch+1}/{epochs} | Batch {batch_idx}/{batches_per_epoch} | Loss {loss.item():.4f} — checkpoint saved")